In [24]:
"""
Convert the subsetted NextGen v2.2 hydrofabric geopackage to the partitioned model json file used in
the KF codebase.

Written by Quinn Russell
"""

import json
import geopandas as gpd

In [25]:
# # read original json format to get partitions
# with open("../1/texas_with_reservoirs_calibrated.json", "r", encoding="utf-8") as f:
#     nwm_models = json.load(f)

In [32]:
# read gpkg to get topology, length, and geometry
gpkg = "/Users/qylee/Downloads/cat-3062147/config/cat-3062147_subset.gpkg"
flowpaths = gpd.read_file(gpkg, layer="flowpaths")
nexus = gpd.read_file(gpkg, layer="nexus")

In [33]:
# Map NWM IDs to NextGen IDs
with open("../1/hf2.2_ref_hf_map.json", "r", encoding="utf-8") as f:
    nwm_to_ngen_mapping = json.load(f)

In [ ]:
# This is not a particularly efficient cell but it should run in about 8 seconds

individual_models = {}

# sorted to improve lookup speed
wb_ids = flowpaths["id"].to_list()
ngen_reach_ids = sorted(wb_ids)

num_reaches = len(ngen_reach_ids)
startnodes = list(range(num_reaches))
endnodes = []

# get endnodes
for reach in ngen_reach_ids:
    to_nexus = flowpaths[flowpaths["id"]==reach]["toid"].values[0]
    next_reach = nexus[nexus["id"]==to_nexus]["toid"].values[0]
    if next_reach not in ngen_reach_ids:
        next_reach = reach # this is how KF handles terminal reaches
    endreach_id = ngen_reach_ids.index(next_reach)
    endnodes.append(endreach_id)

# get lengths
dx = []
for reach in ngen_reach_ids:
    length = flowpaths[flowpaths["id"]==reach]["lengthkm"].values[0] * 1000 # km to m
    dx.append(length)

# get paths
paths = []
for reach in ngen_reach_ids:
    linestring = flowpaths[flowpaths["id"]==reach]["geometry"].values[0]
    try:
        path = [list(pt) for pt in linestring.coords]
    except NotImplementedError:
        path = [0, 0]
    paths.append(path)

stripped_ids = [reach[3:] for reach in ngen_reach_ids]

individual_models["0"] = {
    "name": "0",
    "datetime": "2025-11-30T16:00:00+00:00",
    "timedelta": "P0DT0H15M0S",
    "reach_ids": stripped_ids,
    "startnodes": startnodes,
    "endnodes": endnodes,
    "K": [0] * num_reaches, # dummy K and x values, will be passed in from t-route
    "X": [0] * num_reaches,
    "o_t": [0.0] * num_reaches,
    "dx": dx,
    "paths": paths
}

NotImplementedError: Sub-geometries may have coordinate sequences, but multi-part geometries do not

In [29]:
all_models = {}
for i in [0]:
    all_models[str(i)] = {
        "model": individual_models[str(i)],
        "sinks": [],
        "sources": []
    }

full_partitioned_model = {
    "models": all_models,
    "connections": {}
}

In [30]:
with open("cat-3062147.json", "w", encoding="utf-8") as model_file:
    json.dump(full_partitioned_model, model_file, indent=4)